In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import re

In [2]:
raw_path = Path.cwd().resolve().parent / 'data'/ 'raw'

In [3]:
snapshot_folder = raw_path / "2026-06-23"

calendar_path = snapshot_folder / "calendar.csv.gz"
listings_path = snapshot_folder / "listings.csv.gz"
reviews_path = snapshot_folder / "reviews.csv.gz"

calendar = pd.read_csv(calendar_path, encoding='utf-8', dtype={"listing_id": str})

listings = pd.read_csv(listings_path, encoding='utf-8', dtype={"id": str, "host_id": str, "listing_id": str})

reviews = pd.read_csv(reviews_path, encoding='utf-8', dtype={"id": str, "listing_id": str, "reviewer_id": str, "review_id": str})

In [4]:
# Thông tin file Listings

# print((listings.isna().mean().sort_values(ascending=False).head(15))*100)
# print("\n")
# listings.info()

In [5]:
# # Các thống số file Listings
# print(listings["room_type"].value_counts())
# print("Khong co review: ", listings["number_of_reviews"].eq(0).sum())
# print("Thieu price: ", listings["price"].isna().sum())
# listings.groupby("analysis_area")["id"].size().sort_values(ascending=False)

In [6]:
print(len(calendar))
print(len(reviews))
print(reviews["Mẫu LLM"].notna().sum())
print(reviews["Nhãn tay"].notna().sum())

365000
5000
1000
100


In [7]:
rv_to_label = reviews[reviews["Mẫu LLM"].str.strip().str.lower()=='x'][["id", "comments"]]
samples_path = Path.cwd().resolve().parent / 'data'/ 'samples'
samples_path.mkdir(parents=True, exist_ok=True)
save_path = samples_path / 'rv_llm_baseline.csv'
# rv_to_label.to_csv(save_path, index=False, encoding='utf-8')


In [8]:
d = pd.read_csv(save_path, encoding='utf-8')
d.head(10)["comments"].loc[1]

"Paula's flat is really beautiful; it has loads of space and is full of great artwork. It's in a great location, close to transport and shops and within easy walking distance of the old town.\r<br/>Paula and her husband are extremely welcoming hosts and make you feel at home straight away.\r<br/>Highly recommended."

In [9]:
PAT_SENTIMENT_GOOD = r"amazing|excellent|wonderful|fantastic|perfect|great|lovely|awesome|beautiful|superb|outstanding|fabulous|incredible|brilliant|exceptional|enjoyed|loved|pleasant|delightful|recommend|happy|satisfied|would stay again|had a great time|loved our stay|exceeded expectations|everything was perfect"
PAT_ACCESS_GOOD = r"central|city center|city centre|well located|convenient|accessible|walkable|walking distance|close to|near|nearby|easy access|public transport|metro nearby|heart of|short walk|minutes walk|easy to reach|well connected"
PAT_QUIET = r"quiet|very quiet|peaceful|silent|tranquil|calm|soundproof|no noise|no disturbance|not noisy|quiet neighborhood|peaceful nights|no street noise|well insulated|couldn't hear anything"

PAT_SENTIMENT_BAD = r"terrible|awful|horrible|bad|poor|worst|disappointing|disappointed|unpleasant|unacceptable|frustrating|uncomfortable|dirty|regret|problem|issues|complaint|avoid|rude|would not recommend|never again|not worth it|wouldn't stay again|did not enjoy"
PAT_ACCESS_BAD = r"far from|far away|remote|isolated|inconvenient|difficult to reach|hard to reach|long walk|too far|not central|not accessible|difficult access|far from metro|far from center"
PAT_NOISY = r"noisy|very noisy|too noisy|loud|very loud|traffic noise|street noise|road noise|construction noise|loud music|disturbed sleep|noise at night|loud neighbors|noise problem|unbearable noise"

PAT_SENTIMENT_NEUTRAL = r"okay|ok|average|fine|fair|acceptable|normal|ordinary|nothing special|so-so|could be better|as expected"
PAT_ACCESS_NEUTRAL = r"location was okay|location was fine|reasonable location|location was acceptable|not too far|not too close|moderate distance|location was alright"
PAT_NOISE_NEUTRAL = r"some noise|a little noisy|slightly noisy|bit noisy|minor noise|not too noisy|manageable noise|tolerable noise|noise was okay|noise was fine|noise was acceptable|didn't mind the noise"

def baseline_test(rv_to_label): 
    rv = rv_to_label.copy()
    sentiment_good = rv["comments"].str.findall(PAT_SENTIMENT_GOOD, flags=re.I)
    nums_of_good_sentiment = sentiment_good.map(lambda x: len(x))

    sentiment_bad = rv["comments"].str.findall(PAT_SENTIMENT_BAD, flags=re.I)
    nums_of_bad_sentiment = sentiment_bad.map(lambda x: len(x))
    
    sentiment_neutral = rv["comments"].str.findall(PAT_SENTIMENT_NEUTRAL, flags=re.I)
    nums_of_neutral_sentiment = sentiment_neutral.map(lambda x: len(x))
    
    access_good = rv["comments"].str.contains(PAT_ACCESS_GOOD, flags=re.I)
    access_bad = rv["comments"].str.contains(PAT_ACCESS_BAD, flags=re.I)
    access_neutral = rv["comments"].str.contains(PAT_ACCESS_NEUTRAL, flags=re.I)
    
    noise_quiet = rv["comments"].str.contains(PAT_QUIET, flags=re.I)
    noise_noisy = rv["comments"].str.contains(PAT_NOISY, flags=re.I)
    noise_neutral = rv["comments"].str.contains(PAT_NOISE_NEUTRAL, flags=re.I)

    sentiment_score = nums_of_good_sentiment - nums_of_bad_sentiment
    rv["baseline_general_sentiment"] = ""
    rv.loc[sentiment_score > 0, "baseline_general_sentiment"] = "Positive"
    rv.loc[sentiment_score < 0, "baseline_general_sentiment"] = "Negitive"
    rv.loc[sentiment_score == 0, "baseline_general_sentiment"] = "Mixed" 

    rv["baseline_accessibility"] = "Not mentioned"
    rv.loc[access_good, "baseline_accessibility"] = "Highly accessible"
    rv.loc[access_bad, "baseline_accessibility"] = "Poor access"
    rv.loc[access_neutral, "baseline_accessibility"] = "Neutral"

    rv["baseline_noise"] = "Not mentioned"
    rv.loc[noise_quiet, "baseline_noise"] = "Quiet"
    rv.loc[noise_noisy, "baseline_noise"] = "Noisy"
    rv.loc[noise_neutral, "baseline_noise"] = "Acceptable"

    return rv

In [31]:
import sys
import yaml
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from src import llm

config_path = Path.cwd().resolve().parent / "configs" / "porto.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

In [32]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8')
labeled = llm.llm(nhan_tay, config_path)

Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/processed/llm_cache.jsonl
Luu thanh cong ra file data/

In [33]:
llm = labeled.add_prefix("llm_")
llm = llm.rename(columns={"llm_comments": "comments"})
so_sanh = nhan_tay.merge(llm, how='left', on="comments")
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].sum()

id                                                     2913528046667458522
comments                 Marco’s place was in what felt like a quieter ...
general_sentiment                                MixedMixedMixedMixedMixed
accessibility            Highly accessibleNot mentionedNot mentionedHig...
noise                    QuietNot mentionedNot mentionedQuietNot mentioned
llm_general_sentiment             PositivePositivePositivePositivePositive
llm_accessibility        Highly accessibleNot mentionedNot mentionedHig...
llm_noise                QuietNot mentionedNot mentionedQuietNot mentioned
dtype: object

In [34]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
11,649084508992197281,This stay was a top for us! The patio was an a...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
15,236426625,"Yes, the place is great, accurate, clean and i...",Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
25,931713163239242553,Our stay at the apartment was fantastic! The l...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
29,160065247,We had a great stay at Eduardo's apartment. It...,Positive,Neutral,Not mentioned,Positive,Highly accessible,Not mentioned
36,678147692459333445,This is a fantastic AirBnB property in a great...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
37,1409350478759225419,The unit was clean and neat. Check-in was easy...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
39,508116808,Great location to explore Porto. The apartmen...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
41,529802597,Incredible flat! The private roof terrace has ...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
44,859264500248633249,Staying at Dino & Ines’s Airbnb is highly reco...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
47,1482588535380711857,Best Location in town !!! - Very clean and Coz...,Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned


In [35]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]].loc[58, "comments"]

'We had the pleasure of staying at Nuno and Anita’s apartment, it was by far the best location we stayed at during our month long stay in Portugal.  Nuno and Anita were so thoughtful and made sure we had everything we needed when we arrived.  There even was a crib, high chair, and baby accessories available as we were travelling with an infant.  Communication was excellent, the apartment was extremely clean and well thought out.  We only stayed for 2 nights which was the biggest regret during the stay.  We will definitely stay here again when we visit Porto again.   Highly recommended!!!'

In [36]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
5,603462084469947530,We enjoyed our stay here. Apartment is nice an...,Positive,Highly accessible,Noisy,Positive,Highly accessible,Acceptable
28,1165817981166064555,Very experienced and efficient host. Apartment...,Positive,Not mentioned,Not mentioned,Positive,Not mentioned,Quiet
55,614990524647268194,"Perfect location, most of the atractions are o...",Positive,Highly accessible,Not mentioned,Positive,Highly accessible,Quiet
63,1504315755888942628,We absolutely loved the apartment. It looks ju...,Positive,Highly accessible,Acceptable,Positive,Highly accessible,Quiet
95,1273139733737543268,We loved staying at Daniel's place in Porto. ...,Positive,Highly accessible,Not mentioned,Positive,Highly accessible,Quiet


In [ ]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]].loc[17, "comments"]

'Hugo’s place was very easy to find and he provided helpful instructions on how to get to it. It was located in a great neighbourhood and located within close proximity to various restaurants and monuments. The apartment was fairly well maintained and not too loud at night which was great! We would recommend!'

In [37]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
3,358702512,Marco’s place was in what felt like a quieter ...,Mixed,Highly accessible,Quiet,Positive,Highly accessible,Quiet
31,811461050575532798,"As other reviews mentioned, there’s a funky sm...",Mixed,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
38,1207951934373745373,"This Airbnb was beautiful, it was very spaciou...",Mixed,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
53,513866922,"Very good location, near by train station, exa...",Mixed,Highly accessible,Quiet,Positive,Highly accessible,Quiet
86,894115060845610917,The place was very clean and modern. Great loc...,Mixed,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned


In [29]:
 so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].loc[3,"comments"]

'Marco’s place was in what felt like a quieter place in Porto,  but everything is walkable! Easy to get around to stores, the churches, and train stations. Was a good stay!'